# Supervised experiment — ML owner

Populated skeleton, not a completed model run. Configure the permitted Databricks workspace/package and fill the evaluation/logging cells. The API consumes frozen outputs. Follow PRD §7–8 and docs/databricks.md.

In [ ]:
# TODO: configure permitted repository/volume path and package import.
from pathlib import Path
from hf_followup.data.ingest import ingest_csv
from hf_followup.ml.training import training_frame, fixed_split, candidate_pipelines, development_folds, ALPHA
SOURCE_CSV = Path("/Workspace/Repos/TEAM/REPOSITORY/data/heart_failure_clinical_records.csv")
# ingestion = ingest_csv(SOURCE_CSV)

In [ ]:
# Uncomment after source verification. Save split IDs/hash before tuning.
# x, y = training_frame(ingestion.cohort, ingestion.outcomes)
# split = fixed_split(ingestion.cohort, ingestion.outcomes)
# development_ids, test_ids = split["development_ids"], split["test_ids"]
# assert len(development_ids) == 239 and len(test_ids) == 60
# assert not set(development_ids) & set(test_ids)
# TODO: persist split_manifest.json with source hash and software versions.

In [ ]:
# Constructors fit no data. Scaling stays inside each fold-local pipeline.
pipelines = candidate_pipelines()
folds = development_folds()
print({name: str(pipeline) for name, pipeline in pipelines.items()})
# TODO: declare compact hyperparameter grid and practical-equivalence selection rule
# before results. x never contains time/DEATH_EVENT.

## TODO: development CV and selection

Evaluate age/points references, logistic regression and random forest on identical five validation folds. Clone/refit full pipelines on fold training only. K_eval = ceil((25/299) × validation N). Save precision@K, capture@K, lift, ROC-AUC, average precision, Brier (probabilities), per-fold values and variability. Select by mean capacity capture/precision, then discrimination/stability; prefer the simpler explainable pipeline when practically indistinguishable. Log every candidate to Databricks MLflow. Never use the held-out test to select/tune.

In [ ]:
# TODO: implement CV here using sklearn.base.clone and fold-local fit.
# for train_idx, val_idx in folds.split(x.loc[development_ids], y.loc[development_ids]):
#     train_ids = [development_ids[i] for i in train_idx]
#     validation_ids = [development_ids[i] for i in val_idx]
#     fitted = clone(candidate).fit(x.loc[train_ids], y.loc[train_ids])
#     scores = fitted.predict_proba(x.loc[validation_ids])[:, 1]
#     ... capacity metrics with shared hash tie policy ...
#     ... age/points references on the same validation IDs ...
# Persist fold IDs/results, parameters, seed, versioned selection rationale.

## TODO: frozen test and publication

Refit selected pipeline on development only. Freeze; evaluate once on 60 untouched test records with K=6. Log complete pipeline, signature/input example, immutable run URI, split, metadata, reports and compatible versions. Produce model_metadata.json, split_manifest.json, evaluation_report.json and trusted offline artifact if needed; verify reload scores/order. Publish Delta predictions tagged development_in_sample/held_out_test. Logistic contribution = fitted-scaled value × coefficient in log-odds plus intercept. Forest P0 = recorded indicators; local attribution unavailable. No global importances passed as local attribution, or uncalibrated clinical percentages. Report disappointing results honestly.